# core

> Gateway naming, startup delivery, and MCP sessions on rustygate gateways

In [ ]:
#| default_exp core

[Rustygate](https://github.com/AnswerDotAI/rustygate) hosts kernels and serves MCP requests at `POST /mcp`. Clikernel runs alongside it for the duration of a conversation. It routes the LLM client's stdio MCP requests to gateways.

This module connects to those gateways. `gateways.toml` gives them names. `session_defaults` supplies kernel startup code and, for a local gateway, the conversation's working directory and environment. It sends `startup.py` as source, so it can run on another machine.

`Gateway` represents one MCP session on one gateway. `default_gateway` connects to the local gateway or starts a child process if it cannot connect. The caller must stop any returned child. The MCP router does this when the conversation ends.

Rustygate tracks the session's current kernel and which kernels it created. Ending a session stops its autoclose kernels. It leaves other kernels running. Stopping an owned gateway process stops all kernels in that process.


In [ ]:
#| export
import os, tomllib, httpx
from fastcore.utils import *
from fastcore.xdg import xdg_config_home
from mcpmini.core import HTTPTransport, jreq
from rustygate.tools import start_gateway
from clikernel import __version__

In [ ]:
from fastcore.test import *
from IPython.display import Markdown
import tempfile, re


## Configuration

In [ ]:
#| export
DEFAULT_URL = 'http://127.0.0.1:8787'

def cfg_dir():
    "Return the clikernel configuration directory."
    return xdg_config_home()/'clikernel'

def gateways(cfgdir=None):
    "Read named gateways as `{name: {url, token | token_env, verify}}` from `gateways.toml`."
    p = (Path(cfgdir) if cfgdir else cfg_dir())/'gateways.toml'
    return tomllib.loads(p.read_text()).get('gateways', {}) if p.exists() else {}

def resolve(host='', cfgdir=None):
    "Resolve an empty host, URL, or configured gateway name to `(url, token, verify)`."
    if not host: return os.environ.get('CLIKERNEL_HOST', DEFAULT_URL), os.environ.get('CLIKERNEL_TOKEN'), True
    if '://' in host: return host, os.environ.get('CLIKERNEL_TOKEN'), True
    cfgdir = Path(cfgdir) if cfgdir else cfg_dir()
    g = gateways(cfgdir).get(host)
    if g is None: raise ValueError(f"unknown gateway {host!r}: not a URL, and not in {cfgdir/'gateways.toml'}")
    return g['url'], g.get('token') or os.environ.get(g.get('token_env','')) or None, g.get('verify', True)


Clikernel reads its configuration from `$XDG_CONFIG_HOME/clikernel/`, or `~/.config/clikernel/` when the variable is unset. Both files are optional:

- `startup.py` runs in each Python kernel a session creates.
- `gateways.toml` assigns names and connection settings to gateways.

Use a gateway name to keep authentication tokens out of tool arguments. Tool arguments persist in conversation transcripts. A named gateway can read its token from `token_env` or store it directly as `token`. A nonempty `token` takes precedence.

`resolve` returns `(url, token, verify)`. An empty `host` uses `CLIKERNEL_HOST`, defaulting to `http://127.0.0.1:8787`. A URL passes through unchanged. Both forms use `CLIKERNEL_TOKEN` and enable certificate verification. Any other value names an entry in `gateways.toml`. Unknown names raise `ValueError`.

For example, this entry connects to a gateway using a token from the environment:

    [gateways.solveit]
    url = "https://solveit.example.com/gate"
    token_env = "SOLVEIT_TOKEN"
    verify = false   # accept a self-signed certificate (e.g. rustygate --tls)

`verify = false` disables TLS certificate verification. Omit it to use the default, `true`.


In [ ]:
tmp = tempfile.TemporaryDirectory()
cfgd = Path(tmp.name)
(cfgd/'gateways.toml').write_text(r'''
[gateways.solveit]
url = "https://s.example.com/gate"
token = "T"
verify = false
''')
test_eq(resolve(), (DEFAULT_URL, os.environ.get('CLIKERNEL_TOKEN'), True))
test_eq(resolve('http://h:1/p'), ('http://h:1/p', os.environ.get('CLIKERNEL_TOKEN'), True))
resolved = resolve('solveit', cfgd)
test_eq(resolved, ('https://s.example.com/gate', 'T', False))
resolved


('https://s.example.com/gate', 'T', False)

An unknown gateway name raises `ValueError` and identifies the configuration file to check.

In [ ]:
with expect_fail(ValueError, contains=str(cfgd/'gateways.toml')): resolve('nope', cfgd)

## Startup


Clikernel sends the contents of `startup.py` to the gateway as part of one Python program. The gateway doesn't need access to the file. This keeps the local configuration in control even when the kernel runs on another machine.

The gateway runs the program before user code in each Python kernel the session creates. It first sets `get_ipython().ast_node_interactivity = 'all'`, so every expression result displays. Your `startup.py` runs next and can override this default. During execution, `__file__` contains its path on the client machine. The wrapper deletes `__file__` afterwards. This preserves the behavior of clikernel v1's `%run -i`. Startup output appears in the reply that announces the new kernel.

Non-Python kernels never run this Python startup program. The gateway still applies working-directory and environment defaults. `create(kernel=...)` selects the implementation; `exec` runs code in the selected kernel. The router forwards both calls unchanged.


In [ ]:
#| export
def _startup_src(src, path):
    "Wrap startup source with `__file__` set to its path during execution and deleted afterwards."
    return f'''__file__ = {str(path)!r}
try: exec(compile({src!r}, __file__, 'exec'))
finally: del __file__'''

`startup_src` returns the startup program: the Python defaults, then `startup.py`. `session_defaults` includes that source in the gateway's initialization settings:

In [ ]:
#| export
def startup_src(cfgdir=None):
    "Build Python defaults and user startup, in that order."
    d = Path(cfgdir) if cfgdir else cfg_dir()
    parts = ["get_ipython().ast_node_interactivity = 'all'"]
    if (p := d/'startup.py').exists(): parts.append(_startup_src(p.read_text(), p))
    return '\n'.join(parts)

def session_defaults(cfgdir=None, quiet=False, local=True):
    "Return startup and quiet settings for rustygate initialization. Local sessions also include cwd and env."
    d = dict(startup=startup_src(cfgdir), quiet=quiet)
    if local:
        d['cwd'] = os.getcwd()
        d['env'] = dict(os.environ)
    return d

`session_defaults` returns the fields for the `rustygate` extension to MCP initialization. It always includes `startup` and `quiet`. With `local=True`, it also includes the current working directory and a copy of the environment.

The router uses `local=False` for named hosts. This avoids sending local paths and environment variables to a different machine. Startup source still goes to that gateway.

The examples below use a startup file that defines `base` and prints its filename.

In [ ]:
startup = r'''
base = 42
print("ready, file", __file__.rsplit("/", 1)[-1])
'''
_ = (cfgd/'startup.py').write_text(startup)

The local defaults include working directory and environment. Named gateways receive startup source and the quiet flag without those machine-specific fields.

In [ ]:
d = session_defaults(cfgd)
assert 'ready, file' in d['startup']
remote = session_defaults(cfgd, local=False)
test_eq(remote.keys(), {'startup', 'quiet'})
{'local': sorted(d), 'remote': sorted(remote)}

{'local': ['cwd', 'env', 'quiet', 'startup'], 'remote': ['quiet', 'startup']}

## The gateway session

`Gateway` opens an MCP session with `initialize`. Rustygate tracks the current kernel and autoclose policy. The client manages the connection and request ids, not kernel ownership.

Authentication uses a bearer token. `verify=False` disables TLS certificate verification, including for self-signed certificates.

In [ ]:
#| export
class Gateway:
    "Open an MCP session on rustygate, call its tools, and end the session with DELETE."
    def __init__(
        self,
        url,          # The gateway base URL, e.g. 'http://127.0.0.1:8787'
        token=None,   # Gateway auth token, sent as a bearer token
        verify=True,  # Verify TLS certificates?
    ):
        client = httpx.AsyncClient(verify=verify, timeout=httpx.Timeout(None, connect=10))
        self.url,self._id = url,0
        self.tr = HTTPTransport(f"{url.rstrip('/')}/mcp", token=token, http_client=client)

    async def rpc(self, method, **params):
        "Send a JSON-RPC request and return its result. Raise `RuntimeError` for protocol errors."
        self._id += 1
        r = await self.tr.send(jreq(method, self._id, **params))
        if 'error' in r: raise RuntimeError(f"{r['error']['code']}: {r['error']['message']}")
        return r['result']

    async def initialize(self, defaults=None):
        "Open the MCP session with `defaults` in the `rustygate` extension. Return self."
        await self.tr.start()
        self.info = await self.rpc('initialize', protocolVersion='2025-11-25', capabilities={},
            clientInfo=dict(name='clikernel', version=__version__), rustygate=defaults or {})
        self.tr.proto = self.info['protocolVersion']
        await self.tr.send(jreq('notifications/initialized'))
        return self

Open a session on a disposable gateway using the configuration above. Initialization exchanges MCP protocol information without creating a kernel.

In [ ]:
g = start_gateway()
os.environ['CLIK_DEMO'] = 'via-defaults'
gw = await Gateway(g.url).initialize(session_defaults(cfgd))
assert 'protocolVersion' in gw.info
gw.info['serverInfo']

{'name': 'rustygate', 'version': '0.1.20'}

`call` returns the complete tool result, including non-text content. `text` joins the text blocks and raises `RuntimeError` for an `isError` result.

In [ ]:
#| export
@patch
async def tools(self:Gateway): return (await self.rpc('tools/list'))['tools']
@patch
async def call(self:Gateway, name, **args): return await self.rpc('tools/call', name=name, arguments=args)

@patch
async def text(self:Gateway, name, **args):
    "Join a tool reply's text blocks. Raise `RuntimeError` for `isError` replies."
    r = await self.call(name, **args)
    t = ''.join(c.get('text','') for c in r['content'] if c['type'] == 'text')
    if r.get('isError'): raise RuntimeError(t)
    return t

`create(kernel='py')` makes an unbound Python kernel current. The gateway runs startup before returning the kernel id and banner:

In [ ]:
banner = await gw.text('create', kernel='py')
kid = re.search(r'created kernel (\w+)', banner).group(1)
assert 'ready, file startup.py' in banner
Markdown(banner)

created kernel da209d4899ca42278169428c55a3d202 kernel=ipymini language=python
Exec runs Python/IPython cells; magics such as %%bash work as written.ready, file startup.py


The kernel retains `base` from startup. Python cells display every expression result by default:


In [ ]:
base = await gw.text('exec', code=r'''base
base+1''')
assert '42' in base and '43' in base
PrettyString(base)


<execute_result>
42
</execute_result>
<execute_result>
43
</execute_result>

The kernel inherits the conversation's working directory and environment. `CLIK_DEMO` was set after starting the gateway; it reaches the kernel through session defaults.

In [ ]:
cwd = await gw.text('exec', code='import os; os.getcwd()')
env_value = await gw.text('exec', code="os.environ['CLIK_DEMO']")
test_eq(cwd, repr(os.getcwd()))
test_eq(env_value, "'via-defaults'")
{'cwd': cwd, 'CLIK_DEMO': env_value}

{'cwd': "'/Users/jhoward/aai-ws/clikernel/nbs'", 'CLIK_DEMO': "'via-defaults'"}

IPython cell magics work through the same execution tool. `%%bash` runs a shell cell:

In [ ]:
hi = await gw.text('exec', code="%%bash\necho hi")
test_eq(hi, 'hi\n')
hi

'hi\n'

`aclose` sends an HTTP DELETE to end the session. It closes the HTTP connection even if termination fails, then propagates that error. A failed request does not establish that the remote kernels stopped.

In [ ]:
#| export
@patch
async def aclose(self:Gateway):
    "Request session termination and close the HTTP connection."
    try: await self.tr.delete()
    finally: await self.tr.aclose()

Pass `dlgname` to bind a kernel to a dialog name. A new kernel closes with its creating session by default. Pass `autoclose=False` to keep it running after that session ends.

`use_kernel` selects an existing kernel without taking responsibility for its lifetime. Here, a second session selects the first session's unbound Python kernel. Closing the first session stops that kernel even though the second session uses it. The named `demo.ipynb` kernel also stops. Only the kernel created with `autoclose=False` remains:

In [ ]:
assert (await gw.text('create', kernel='py', dlgname='demo.ipynb')).startswith('created kernel ')
kept = re.search(r'kernel (\w+)', await gw.text('create', kernel='py', dlgname='keeper.ipynb', autoclose=False)).group(1)

gw2 = await Gateway(g.url).initialize(session_defaults(cfgd))
assert kid[:8] in await gw2.text('use_kernel', kernel=kid[:8])
test_eq(await gw2.text('exec', code='base'), '42')

await gw.aclose()
listing = await gw2.text('list_kernels')
assert kid not in listing and 'demo.ipynb' not in listing and kept in listing
listing

'7d2d7805b1e64193b236cb90a51e3cbd  alive  kernel=ipymini  language=python  connections=0  dlgname=keeper.ipynb'

`restart` replaces the interpreter but keeps the kernel id. For a kernel this session created, it reruns startup and restores `base`. The later assignment to `y` does not survive.

In [ ]:
assert 'created kernel' in await gw2.text('create', kernel='py', dlgname='r.ipynb')
await gw2.text('exec', code='y = 5')
res = await gw2.text('restart')
assert res.startswith('restarted kernel ') and 'ready' in res
test_eq(await gw2.text('exec', code='base'), '42')
assert 'NameError' in await gw2.text('exec', code='y')
res

'restarted kernel a4215c060636428cad339ceeda5fa260ready, file startup.py\n'

`quiet=True` suppresses startup output without skipping startup. The reply still identifies the new kernel. `base` remains available even though the reply no longer contains the banner:

In [ ]:
gwq = await Gateway(g.url).initialize(session_defaults(cfgd, quiet=True))
qbanner = await gwq.text('create', kernel='py')
assert 'created kernel' in qbanner and 'ready' not in qbanner
test_eq(await gwq.text('exec', code='base'), '42')
await gwq.aclose()
qbanner

'created kernel 924287806991480881f63efa8cf1e385 kernel=ipymini language=python\nExec runs Python/IPython cells; magics such as %%bash work as written.'

## The default gateway

`default_gateway` tries to initialize a session at the default local URL. If connecting raises `httpx.ConnectError`, it starts a rustygate child on a free port and connects there. It returns the initialized `Gateway` and the child process, or `None` instead of a child when it reused a running gateway.

The caller must close the session and stop any returned child. The MCP router does this when the conversation ends. It doesn't stop a gateway that was already running.

A kernel can outlive the conversation only if its gateway also stays running. Start a separate `rustygate` service when you need persistence. Kernels created with autoclose still stop at session end, even on that service. Use `autoclose=False` to keep a newly created kernel. Nothing survives shutdown of a child gateway.

In [ ]:
#| export
async def default_gateway(
    cfgdir=None,  # Config dir for `session_defaults` (the standard one if None)
    quiet=False,  # Keep startup output out of replies?
):
    "Return an initialized `Gateway` and its new child process, or None if it reused a gateway. The caller must stop any child."
    url, token, verify = resolve('', cfgdir)
    d = session_defaults(cfgdir, quiet)
    try: return await Gateway(url, token, verify).initialize(d), None
    except httpx.ConnectError:
        child = start_gateway()
        return await Gateway(child.url).initialize(d), child

First, point `CLIKERNEL_HOST` at our running test gateway. `default_gateway` connects without starting another process:


In [ ]:
os.environ['CLIKERNEL_HOST'] = g.url
gf, child = await default_gateway(cfgd)
assert child is None and gf.url == g.url
await gf.aclose()
gf.url


'http://127.0.0.1:55945'

Now point it at a port with no listener. `default_gateway` starts a child and applies the same startup settings. We close the session and stop that child after checking the reply:

In [ ]:
os.environ['CLIKERNEL_HOST'] = 'http://127.0.0.1:1'
go, child = await default_gateway(cfgd)
assert child is not None and go.url == child.url
banner = await go.text('create', kernel='py')
assert 'created kernel' in banner and 'ready' in banner
await go.aclose()
child.stop()
child.url

'http://127.0.0.1:56150'

In [ ]:
#| hide
await gw2.aclose()
g.stop()
tmp.cleanup()
for k in ('CLIKERNEL_HOST', 'CLIK_DEMO'): os.environ.pop(k, None)

In [ ]:
#|hide
#|eval: false
import nbdev
nbdev.nbdev_export()